# Model_Optimization_and_Evaluation

## 1. Library Import

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    ParameterGrid,
    GridSearchCV,
    StratifiedKFold,
)
from modeling.models import get_models
from modeling.experiment import run_experiment, run_cv_threshold_experiment
from preprocessing.pipeline import build_pipeline
from modeling.metrics import calculate_metrics
from sklearn.model_selection import train_test_split
from data.loader import load_dataset
from imblearn.over_sampling import (
    SMOTE,
    BorderlineSMOTE,
    ADASYN,
)

## 2. Data Load & Split

In [2]:
df = load_dataset("../processed/uci-secom-constant-removed.csv")
df.head()

,Time,0,1,2,3,4,6,7,8,9,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,97.6133,0.1242,1.5005,0.0162,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,102.3433,0.1247,1.4966,-0.0005,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,95.4878,0.1241,1.4436,0.0041,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,104.2367,0.1217,1.4882,-0.0124,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.3967,0.1235,1.5031,-0.0031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [3]:
X = df.drop(columns=["Time", "Pass/Fail"])
y = df["Pass/Fail"]

y = y.replace({
    -1: 0,
     1: 1,
    })

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    )

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")

print("\n")

print("-----Train-----\n", y_train.value_counts(), "\n\n", y_train.value_counts(normalize=True))

print("\n")

print("-----Test-----\n", y_test.value_counts(), "\n\n", y_test.value_counts(normalize=True))

Train : (1253, 474)
Test  : (314, 474)


-----Train-----
 Pass/Fail
0    1170
1      83
Name: count, dtype: int64 

 Pass/Fail
0    0.933759
1    0.066241
Name: proportion, dtype: float64


-----Test-----
 Pass/Fail
0    293
1     21
Name: count, dtype: int64 

 Pass/Fail
0    0.933121
1    0.066879
Name: proportion, dtype: float64


## 3. Missing Value Threshold

In [4]:
missing_thresholds = {
    "0%": 0.00,
    "1%": 0.01,
    "20%": 0.2,
    "60%": 0.6,
    "80%": 0.8,
    "90%": 0.9,
}

In [6]:
missing_results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    models=get_models(),
    experiment_name="Missing",
    methods=missing_thresholds,
    parameter_name="missing_threshold",
    pipeline_kwargs={
        "scaler": "standard",
    },
)

missing_results

threshold  accuracy  precision  \
experiment method model                                                 
Missing    0%     Logistic Regression        0.5  0.922586   0.150000   
                  Random Forest              0.5  0.934557   1.000000   
                  XGBoost                    0.5  0.930567   0.000000   
           1%     Logistic Regression        0.5  0.891460   0.195402   
                  Random Forest              0.5  0.933759   0.000000   
                  XGBoost                    0.5  0.932163   0.250000   
           20%    Logistic Regression        0.5  0.883480   0.188119   
                  Random Forest              0.5  0.933759   0.000000   
                  XGBoost                    0.5  0.931365   0.285714   
           60%    Logistic Regression        0.5  0.885874   0.187500   
                  Random Forest              0.5  0.933759   0.000000   
                  XGBoost                    0.5  0.934557   0.571429   
           80%    Logistic Regression        0.5  0.885076   0.178947   
                  Random Forest              0.5  0.933759   0.000000   
                  XGBoost                    0.5  0.932961   0.428571   
           90%    Logistic Regression        0.5  0.887470   0.177778   
                  Random Forest              0.5  0.933759   0.000000   
                  XGBoost                    0.5  0.933759   0.500000   

                                         recall        f1        f2  \
experiment method model                                               
Missing    0%     Logistic Regression  0.036145  0.058252  0.042614   
                  Random Forest        0.012048  0.023810  0.015015   
                  XGBoost              0.000000  0.000000  0.000000   
           1%     Logistic Regression  0.204819  0.200000  0.202864   
                  Random Forest        0.000000  0.000000  0.000000   
                  XGBoost              0.012048  0.022989  0.014881   
           20%    Logistic Regression  0.228916  0.206522  0.219400   
                  Random Forest        0.000000  0.000000  0.000000   
                  XGBoost              0.024096  0.044444  0.029499   
           60%    Logistic Regression  0.216867  0.201117  0.210280   
                  Random Forest        0.000000  0.000000  0.000000   
                  XGBoost              0.048193  0.088889  0.058997   
           80%    Logistic Regression  0.204819  0.191011  0.199063   
                  Random Forest        0.000000  0.000000  0.000000   
                  XGBoost              0.036145  0.066667  0.044248   
           90%    Logistic Regression  0.192771  0.184971  0.189573   
                  Random Forest        0.000000  0.000000  0.000000   
                  XGBoost              0.036145  0.067416  0.044379   

                                             confusion_matrix  overkill_count  \
experiment method model                                                         
Missing    0%     Logistic Regression   [[1153, 17], [80, 3]]              17   
                  Random Forest          [[1170, 0], [82, 1]]               0   
                  XGBoost                [[1166, 4], [83, 0]]               4   
           1%     Logistic Regression  [[1100, 70], [66, 17]]              70   
                  Random Forest          [[1170, 0], [83, 0]]               0   
                  XGBoost                [[1167, 3], [82, 1]]               3   
           20%    Logistic Regression  [[1088, 82], [64, 19]]              82   
                  Random Forest          [[1170, 0], [83, 0]]               0   
                  XGBoost                [[1165, 5], [81, 2]]               5   
           60%    Logistic Regression  [[1092, 78], [65, 18]]              78   
                  Random Forest          [[1170, 0], [83, 0]]               0   
                  XGBoost                [[1167, 3], [79, 4]]               3   
           80%    Logis

### Observations

- 결측치 기준을 0%로 설정하여 결측치가 조금이라도 있는 데이터를 모두 버렸을 때, 분석에 사용할 수 있는 공정 변수의 개수가 102.4개로 크게 줄어들었습니다.

- 이때 결측치가 있던 변수들이 빠진 자리를 결측 여부를 보여주는 인디케이터 변수들이 채우게 되는데, 두 변수의 편차가 일치하는 안정적인 데이터 구조를 보였습니다.

- 이후 결측치 허용 기준을 1%에서 20%로 넓혀주면 분석에 쓸 수 있는 공정 변수가 각각 368.8개, 442.0개로 빠르게 복원되면서 모델의 전반적인 불량 예측 성능이 향상되는 것을 확인했습니다.

- 하지만 20%를 넘어가면 유효한 변수 개수가 더 늘어나지 않고 성능도 비슷하게 유지되었습니다.

- 인디케이터 변수의 개수가 모든 조건에서 평균 371.6개로 일정한 것은 인디케이터 변수 자체에는 빈 값이 없기 때문에 필터링 과정에서 걸러지지 않고 끝까지 남았기 때문입니다.

- 기본 분류 기준인 0.5를 기준으로 보면, 정상 제품을 불량으로 잘못 판정하는 오검출과 실제 불량을 놓치는 미검출 사이에 뚜렷한 차이가 납니다.

- Logistic Regression은 결측치 기준 20%에서 실제 불량을 가장 잘 잡아내어 미검출을 64건까지 낮췄지만, 정상 제품을 불량으로 오진하는 오검출이 82건으로 가장 많았습니다.

- 반대로 Random Forest와 XGBoost 같은 트리 기반 모델들은 데이터의 불량이 너무 적다 보니 정상 판정으로 치우쳐 불량을 거의 잡아내지 못하는 유출 리스크를 보였습니다.

- 그래도 XGBoost 모델은 결측치 기준 60%에서 오검출을 단 3건으로 막아내며 안정적인 제어 성능을 보여주었습니다.

### Interpretation

- 이러한 관찰을 통해 볼 때, 결측치가 있다고 데이터를 무조건 지우는 0% 기준은 수율 예측에 필요한 중요한 정보를 잃게 만들어 모델 성능을 떨어뜨립니다.

- 반면 결측치 기준을 20%보다 너무 크게 잡는 것은 빈 칸을 채워 넣는 과정에서 오히려 부정확한 노이즈 데이터를 만들어내 변수 개수만 늘어나고 예측력은 더 좋아지지 않는 한계가 있습니다.

- 따라서 정보의 양과 데이터의 정확성을 모두 챙길 수 있는 최적의 결측치 처리 기준은 1%에서 20% 사이로 판단됩니다.
또한 데이터 불균형이 심한 상황에서 기본 예측 기준인 0.5를 그대로 쓰는 것은 현업에서 활용하기 어렵습니다.

- 로지스틱 회귀 모델은 정상 제품을 불량으로 오해하는 비용을 감수하더라도 현장에 불량이 유출되는 것을 엄격하게 막아주는 성향을 보입니다.

- 반대로 트리 모델들은 특별한 가중치 조정이 없다면 정상 제품으로만 예측하려는 편향된 모습을 보입니다.

- 결국 불량을 놓쳐 고객사로 유출되는 품질 비용과 정상 제품을 불량으로 잘못 판정해 다시 검사하느라 낭비되는 생산 비용 사이에서 적절한 균형을 찾는 의사결정이 필요합니다.

### Decision

- Logistic Regression의 최적 설정값은 결측률 임계치 20%로 확정합니다. 공정에서 불량이 밖으로 유출되는 것을 가장 철저히 막아야 하는 상황에 적합하며 데이터의 안정성과 성능이 고르게 확보되는 지점입니다.

- Random Forest의 최적 설정값은 결측률 임계치 60%로 지정합니다. 예측의 잠재력을 나타내는 전체 성능 지표가 가장 우수하게 나왔으며, 향후 분류 기준값을 조정하는 실험의 핵심 후보군으로 선별합니다.

- XGBoost 모델의 최적 설정값은 결측률 임계치 60%로 결정합니다. 오검출을 단 3건으로 막아내어 불필요한 추가 검사 비용을 최소화하는 뛰어난 효율성을 보여주었습니다.

### Candidate

missing-value threshold

Logistic Regression : 0.2  
Random Forest : 0.6  
XGBoost : 0.6

## 4. Variance Treshold

### Logistic Regression

In [9]:
thresholds = {
    "1e-6": 1e-6,
    "1e-4": 1e-4,
    "1e-2": 1e-2,
    "1e-1": 1e-1,
    "1": 1,
    "10": 10,
}

lr_variance_results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    models={"Logistic Regression": get_models()["Logistic Regression"]},
    experiment_name="Variance",
    methods=thresholds,
    parameter_name="variance_threshold",
    pipeline_kwargs={
        "scaler": "standard",
        "missing_threshold": 0.2,
    },
)

lr_variance_results

threshold  accuracy  precision  \
experiment method model                                                 
Variance   1e-6   Logistic Regression        0.5  0.883480   0.188119   
           1e-4   Logistic Regression        0.5  0.883480   0.188119   
           1e-2   Logistic Regression        0.5  0.878691   0.151515   
           1e-1   Logistic Regression        0.5  0.886672   0.195876   
           1      Logistic Regression        0.5  0.923384   0.217391   
           10     Logistic Regression        0.5  0.928970   0.200000   

                                         recall        f1        f2  \
experiment method model                                               
Variance   1e-6   Logistic Regression  0.228916  0.206522  0.219400   
           1e-4   Logistic Regression  0.228916  0.206522  0.219400   
           1e-2   Logistic Regression  0.180723  0.164835  0.174014   
           1e-1   Logistic Regression  0.228916  0.211111  0.221445   
           1      Logistic Regression  0.060241  0.094340  0.070423   
           10     Logistic Regression  0.024096  0.043011  0.029240   

                                             confusion_matrix  overkill_count  \
experiment method model                                                         
Variance   1e-6   Logistic Regression  [[1088, 82], [64, 19]]              82   
           1e-4   Logistic Regression  [[1088, 82], [64, 19]]              82   
           1e-2   Logistic Regression  [[1086, 84], [68, 15]]              84   
           1e-1   Logistic Regression  [[1092, 78], [64, 19]]              78   
           1      Logistic Regression   [[1152, 18], [78, 5]]              18   
           10     Logistic Regression    [[1162, 8], [81, 2]]               8   

                                       escape_count   roc_auc    pr_auc  \
experiment method model                                                   
Variance   1e-6   Logistic Regression            64  0.634157  0.122933   
           1e-4   Logistic Regression            64  0.634157  0.122933   
           1e-2   Logistic Regression            68  0.635187  0.118442   
           1e-1   Logistic Regression            64  0.641345  0.130623   
           1      Logistic Regression            78  0.651642  0.119501   
           10     Logistic Regression            81  0.587746  0.097521   

                                       remaining_features  \
experiment method model                                     
Variance   1e-6   Logistic Regression               812.4   
           1e-4   Logistic Regression               812.4   
           1e-2   Logistic Regression               800.0   
           1e-1   Logistic Regression               744.6   
           1      Logistic Regression               495.2   
           10     Logistic Regression               398.6   

                                       remaining_features_std  \
experiment method model                                         
Variance   1e-6   Logistic Regression               28.464715   
           1e-4   Logistic Regression               28.464715   
           1e-2   Logistic Regression               27.770488   
           1e-1   Logistic Regression               28.576914   
           1      Logistic Regression               24.579666   
           10     Logistic Regression               27.760403   

                                       remaining_original_features  \
experiment method model                                              
Variance   1e-6   Logistic Regression                        440.8   
           1e-4   Logistic Regression                        440.8   
           1e-2   Logistic Regression                        428.4   
           1e-1   Logistic Regression                        373.0   
           1      Logistic Regression                        123.6   
           10     Logistic Regression                         27.0   

                                       remaining_original_featur

### Random Forest

In [11]:
thresholds = {
    "1e-6": 1e-6,
    "1e-4": 1e-4,
    "1e-2": 1e-2,
    "1e-1": 1e-1,
    "1": 1,
    "10": 10,
}

rf_variance_results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    models={"Random Forest": get_models()["Random Forest"]},
    experiment_name="Variance",
    methods=thresholds,
    parameter_name="variance_threshold",
    pipeline_kwargs={
        "scaler": "standard",
        "missing_threshold": 0.6,
    },
)

rf_variance_results

threshold  accuracy  precision  recall   f1  \
experiment method model                                                        
Variance   1e-6   Random Forest        0.5  0.933759        0.0     0.0  0.0   
           1e-4   Random Forest        0.5  0.933759        0.0     0.0  0.0   
           1e-2   Random Forest        0.5  0.933759        0.0     0.0  0.0   
           1e-1   Random Forest        0.5  0.933759        0.0     0.0  0.0   
           1      Random Forest        0.5  0.933759        0.0     0.0  0.0   
           10     Random Forest        0.5  0.931365        0.0     0.0  0.0   

                                  f2      confusion_matrix  overkill_count  \
experiment method model                                                      
Variance   1e-6   Random Forest  0.0  [[1170, 0], [83, 0]]               0   
           1e-4   Random Forest  0.0  [[1170, 0], [83, 0]]               0   
           1e-2   Random Forest  0.0  [[1170, 0], [83, 0]]               0   
           1e-1   Random Forest  0.0  [[1170, 0], [83, 0]]               0   
           1      Random Forest  0.0  [[1170, 0], [83, 0]]               0   
           10     Random Forest  0.0  [[1167, 3], [83, 0]]               3   

                                 escape_count   roc_auc    pr_auc  \
experiment method model                                             
Variance   1e-6   Random Forest            83  0.689744  0.174771   
           1e-4   Random Forest            83  0.689744  0.174771   
           1e-2   Random Forest            83  0.669787  0.141383   
           1e-1   Random Forest            83  0.699912  0.148937   
           1      Random Forest            83  0.619895  0.108138   
           10     Random Forest            83  0.638220  0.131237   

                                 remaining_features  remaining_features_std  \
experiment method model                                                       
Variance   1e-6   Random Forest               820.4               28.464715   
           1e-4   Random Forest               820.4               28.464715   
           1e-2   Random Forest               808.0               27.770488   
           1e-1   Random Forest               751.4               28.527881   
           1      Random Forest               500.2               24.579666   
           10     Random Forest               398.6               27.760403   

                                 remaining_original_features  \
experiment method model                                        
Variance   1e-6   Random Forest                        448.8   
           1e-4   Random Forest                        448.8   
           1e-2   Random Forest                        436.4   
           1e-1   Random Forest                        379.8   
           1      Random Forest                        128.6   
           10     Random Forest                         27.0   

                                 remaining_original_features_std  \
experiment method model                                            
Variance   1e-6   Random Forest                         2.400000   
           1e-4   Random Forest                         2.400000   
           1e-2   Random Forest                         1.959592   
           1e-1   Random Forest                         2.481935   
           1      Random Forest                         5.122499   
           10     Random Forest                         1.897367   

                                 added_indicator_features  \
experiment method model                                     
Variance   1e-6   Random Forest                     371.6   
           1e-4   Random Forest                     371.6   
           1e-2   Random Forest                     371.6   
           1e-1   Random Forest                     371.6   
           1      Random Forest                     371.6   
           10     Random Forest                     371.6   

                                 a

### XGBoost

In [ ]:
thresholds = {
    "1e-6": 1e-6,
    "1e-4": 1e-4,
    "1e-2": 1e-2,
    "1e-1": 1e-1,
    "1": 1,
    "10": 10,
}

xg_variance_results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    models={"XGBoost": get_models()["XGBoost"]},
    experiment_name="Variance",
    methods=thresholds,
    parameter_name="variance_threshold",
    pipeline_kwargs={
        "scaler": "standard",
        "missing_threshold": 0.6,
    },
)

xg_variance_results

threshold  accuracy  precision    recall        f1  \
experiment method model                                                         
Variance   1e-6   XGBoost        0.5  0.934557   0.571429  0.048193  0.088889   
           1e-4   XGBoost        0.5  0.934557   0.571429  0.048193  0.088889   
           1e-2   XGBoost        0.5  0.934557   0.571429  0.048193  0.088889   
           1e-1   XGBoost        0.5  0.934557   0.600000  0.036145  0.068182   
           1      XGBoost        0.5  0.932163   0.000000  0.000000  0.000000   
           10     XGBoost        0.5  0.931365   0.200000  0.012048  0.022727   

                                 f2      confusion_matrix  overkill_count  \
experiment method model                                                     
Variance   1e-6   XGBoost  0.058997  [[1167, 3], [79, 4]]               3   
           1e-4   XGBoost  0.058997  [[1167, 3], [79, 4]]               3   
           1e-2   XGBoost  0.058997  [[1167, 3], [79, 4]]               3   
           1e-1   XGBoost  0.044510  [[1168, 2], [80, 3]]               2   
           1      XGBoost  0.000000  [[1168, 2], [83, 0]]               2   
           10     XGBoost  0.014837  [[1166, 4], [82, 1]]               4   

                           escape_count   roc_auc    pr_auc  \
experiment method model                                       
Variance   1e-6   XGBoost            79  0.661786  0.169610   
           1e-4   XGBoost            79  0.661786  0.169610   
           1e-2   XGBoost            79  0.650098  0.182345   
           1e-1   XGBoost            80  0.702636  0.193796   
           1      XGBoost            83  0.649449  0.117710   
           10     XGBoost            82  0.597817  0.098059   

                           remaining_features  remaining_features_std  \
experiment method model                                                 
Variance   1e-6   XGBoost               820.4               28.464715   
           1e-4   XGBoost               820.4               28.464715   
           1e-2   XGBoost               808.0               27.770488   
           1e-1   XGBoost               751.4               28.527881   
           1      XGBoost               500.2               24.579666   
           10     XGBoost               398.6               27.760403   

                           remaining_original_features  \
experiment method model                                  
Variance   1e-6   XGBoost                        448.8   
           1e-4   XGBoost                        448.8   
           1e-2   XGBoost                        436.4   
           1e-1   XGBoost                        379.8   
           1      XGBoost                        128.6   
           10     XGBoost                         27.0   

                           remaining_original_features_std  \
experiment method model                                      
Variance   1e-6   XGBoost                         2.400000   
           1e-4   XGBoost                         2.400000   
           1e-2   XGBoost                         1.959592   
           1e-1   XGBoost                         2.481935   
           1      XGBoost                         5.122499   
           10     XGBoost                         1.897367   

                           added_indicator_features  \
experiment method model                               
Variance   1e-6   XGBoost                     371.6   
           1e-4   XGBoost                     371.6   
           1e-2   XGBoost                     371.6   
           1e-1   XGBoost                     371.6   
           1      XGBoost                     371.6   
           10     XGBoost                     371.6   

                           added_indicator_features_std  
experiment method model                                  
Variance   1e-6   XGBoost                     29.131426  
           1e-4   XGBoost                     29.131426  
           1e-2   XGBoost           

### Observations

- 결측치 처리 기준을 고정한 상태에서 분산 임계값의 변화에 따른 성능 변화를 관찰한 결과, 임계값이 커질수록 불량 예측에 사용되는 공정 변수의 개수가 점차 감소하는 흐름을 보였습니다.

- 임계치 1e-6부터 1e-1 구간까지는 변수가 수십 개 단위로 완만하게 줄어들다가, 임계치 1을 기점으로 분석에 사용되는 변수 개수가 급격히 줄어드는 현상이 나타났습니다.

- Logistic Regression의 경우, 분산 임계값 1e-1 조건에서 최종 변수가 744.6개(원본 변수 373.0개)로 정제되었을 때 불량 예측 성능인 ROC AUC가 0.6413으로 이전 단계보다 상승했습니다. 또한 정상 제품을 불량으로 잘못 진단하는 오검출 수치도 기존 82건에서 78건으로 줄어들며 진단 효율성이 함께 개선되었습니다.

- Random Forest와 XGBoost 모델 역시 분산 임계값 1e-1 조건에서 가장 이상적인 성능을 나타냈습니다. Random Forest의 ROC AUC 성능은 0.6999까지 향상되었고, XGBoost의 ROC AUC 성능은 0.7026을 기록하여 이번 실험 조건 중 가장 우수한 수치를 보여주었습니다. 특히 XGBoost는 오검출을 단 2건으로 크게 줄이면서도 안정적인 지표를 유지했습니다.

- 반면 분산 임계값을 1 이상으로 너무 높게 설정하면 분석에 쓰이는 원본 변수의 개수가 120개 수준으로 과도하게 버려지게 됩니다. 이 조건에서는 세 모델 모두 불량을 판정하는 리콜 지표가 급격히 무너지거나, 전반적인 성능 지표가 0.5에서 0.6 수준으로 크게 떨어지는 현상이 일관되게 관찰되었습니다.

### Interpretation

- 이러한 관찰 결과를 바탕으로 분석해 보면, 분산 임계값을 1e-1 수준으로 적용했을 때 수율 예측에 영향이 없고 미세한 흔들림만 있는 저분산 노이즈 변수들이 파이프라인을 통해 효과적으로 걸러진 것으로 해석됩니다. 데이터의 세부적인 정보가 없는 상태에서는 매우 조심스럽게 접근해야 하나 전체 변수의 개수가 감소했음에도 불구하고 예측 성능이 오히려 보존되거나 향상된 것이 근거가 된다고 생각합니다.

- 그러나 임계값을 1 이상으로 과도하게 높이는 것은 비록 분산 자체는 작더라도 실제 공정에서 불량을 가려내는 데 핵심적인 역할을 수행하던 유용한 공정 정보까지 같이 지워버리는 결과를 초래한 것으로 보입니다. 이로 인해 모델이 불량 패턴을 학습하는 데 필요한 최소한의 데이터 균형이 깨지게 되었습니다.

- 결과적으로 데이터의 크기를 줄여 연산의 효율을 높이면서도 수율을 예측하는 모델의 신뢰성을 지킬 수 있는 저분산 제거의 가장 적절한 통제 기준선은 1e-1 구간인 것으로 분석됩니다.

### Decision

- Logistic Regression의 최적 설정값은 분산 임계치 1e-1로 결정합니다. 실제 불량을 유출시키지 않는 방어력(Recall 0.2289)을 원본 데이터 수준으로 고스란히 유지하면서, 오검출을 78건으로 낮춰 최적의 효율을 보여준 지점입니다.

- Random Forest의 최적 설정값은 분산 임계치 1e-1로 결정합니다. 잠재적 분류 성능을 나타내는 ROC AUC 수치가 0.6999로 가장 우수하게 확인되었기에 해당 데이터 구성을 최종 후보로 선별합니다.

- XGBoost 모델의 최적 설정값 역시 분산 임계치 1e-1로 결정합니다. 전체 조건 중에서 가장 높은 ROC AUC(0.7026) 성능을 보여주었으며, 불필요한 재검사 리소스를 최소화할 수 있도록 오검출을 단 2건이면서 불량 3건을 검출하여 가장 우수한 밸런스를 보였습니다.

### Candidate

Logistic Regression : 1e-1  
Random Forest : 1e-1  
XGBoost : 1e-1

## 5. Correlation Filtering & Clustering

### Logistic Regression

In [14]:
joint_tuning_grid = {
    0.90: [0.12, 0.15, 0.20],
    0.95: [0.07, 0.10, 0.15],
    0.98: [0.03, 0.05, 0.10],
    0.99: [0.02, 0.04, 0.08],
}

result_frames = []

for correlation_threshold, cluster_thresholds in joint_tuning_grid.items():

    cluster_methods = {
        str(cluster_threshold): cluster_threshold
        for cluster_threshold in cluster_thresholds
    }

    result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        models={"Logistic Regression": get_models()["Logistic Regression"]},
        experiment_name="Correlation + Clustering",
        methods=cluster_methods,
        parameter_name="cluster_distance_threshold",
        pipeline_kwargs={
            "imputer": "mean",
            "scaler": "standard",
            "missing_threshold": 0.2,
            "variance_threshold": 0.1,
            "correlation_threshold": correlation_threshold,
        },
    ).reset_index()

    result["correlation_threshold"] = correlation_threshold
    result["cluster_distance_threshold"] = (
        result["method"].astype(float)
    )

    result_frames.append(result)

lr_corr_clu_results = (
    pd.concat(result_frames, ignore_index=True)
    .drop(columns="method")
    .set_index(
        [
            "experiment",
            "correlation_threshold",
            "cluster_distance_threshold",
            "model",
        ]
    )
    .sort_index()
)

lr_corr_clu_results

threshold  \
experiment               correlation_threshold cluster_distance_threshold model                            
Correlation + Clustering 0.90                  0.12                       Logistic Regression        0.5   
                                               0.15                       Logistic Regression        0.5   
                                               0.20                       Logistic Regression        0.5   
                         0.95                  0.07                       Logistic Regression        0.5   
                                               0.10                       Logistic Regression        0.5   
                                               0.15                       Logistic Regression        0.5   
                         0.98                  0.03                       Logistic Regression        0.5   
                                               0.05                       Logistic Regression        0.5   
                                               0.10                       Logistic Regression        0.5   
                         0.99                  0.02                       Logistic Regression        0.5   
                                               0.04                       Logistic Regression        0.5   
                                               0.08                       Logistic Regression        0.5   

                                                                                               accuracy  \
experiment               correlation_threshold cluster_distance_threshold model                           
Correlation + Clustering 0.90                  0.12                       Logistic Regression  0.885874   
                                               0.15                       Logistic Regression  0.885874   
                                               0.20                       Logistic Regression  0.894653   
                         0.95                  0.07                       Logistic Regression  0.885874   
                                               0.10                       Logistic Regression  0.883480   
                                               0.15                       Logistic Regression  0.888268   
                         0.98                  0.03                       Logistic Regression  0.889864   
                                               0.05                       Logistic Regression  0.885874   
                                               0.10                       Logistic Regression  0.885874   
                         0.99                  0.02                       Logistic Regression  0.893057   
                                               0.04                       Logistic Regression  0.890662   
                                               0.08                       Logistic Regression  0.886672   

                                                                                               precision  \
experiment               correlation_threshold cluster_distance_threshold model                            
Correlation + Clustering 0.90                  0.12                       Logistic Regression   0.159091   
                                               0.15                       Logistic Regression   0.173913   
                                               0.20                       Logistic Regression   0.204819   
                         0.95                  0.07                       Logistic Regression   0.166667   
                                               0.10                       Logistic Regression   0.153846   
                                               0.15                       Logistic Regression   0.193548   
                         0.98                  0.03                       Logistic Regression   0.197802   
                                               0.05                       Logistic Regression   0.166667   
   

### Random Forest

In [15]:
joint_tuning_grid = {
    0.90: [0.12, 0.15, 0.20],
    0.95: [0.07, 0.10, 0.15],
    0.98: [0.03, 0.05, 0.10],
    0.99: [0.02, 0.04, 0.08],
}

result_frames = []

for correlation_threshold, cluster_thresholds in joint_tuning_grid.items():

    cluster_methods = {
        str(cluster_threshold): cluster_threshold
        for cluster_threshold in cluster_thresholds
    }

    result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        models={"Random Forest": get_models()["Random Forest"]},
        experiment_name="Correlation + Clustering",
        methods=cluster_methods,
        parameter_name="cluster_distance_threshold",
        pipeline_kwargs={
            "imputer": "mean",
            "scaler": "standard",
            "missing_threshold": 0.6,
            "variance_threshold": 0.1,
            "correlation_threshold": correlation_threshold,
        },
    ).reset_index()

    result["correlation_threshold"] = correlation_threshold
    result["cluster_distance_threshold"] = (
        result["method"].astype(float)
    )

    result_frames.append(result)

rf_corr_clu_results = (
    pd.concat(result_frames, ignore_index=True)
    .drop(columns="method")
    .set_index(
        [
            "experiment",
            "correlation_threshold",
            "cluster_distance_threshold",
            "model",
        ]
    )
    .sort_index()
)

rf_corr_clu_results

threshold  \
experiment               correlation_threshold cluster_distance_threshold model                      
Correlation + Clustering 0.90                  0.12                       Random Forest        0.5   
                                               0.15                       Random Forest        0.5   
                                               0.20                       Random Forest        0.5   
                         0.95                  0.07                       Random Forest        0.5   
                                               0.10                       Random Forest        0.5   
                                               0.15                       Random Forest        0.5   
                         0.98                  0.03                       Random Forest        0.5   
                                               0.05                       Random Forest        0.5   
                                               0.10                       Random Forest        0.5   
                         0.99                  0.02                       Random Forest        0.5   
                                               0.04                       Random Forest        0.5   
                                               0.08                       Random Forest        0.5   

                                                                                         accuracy  \
experiment               correlation_threshold cluster_distance_threshold model                     
Correlation + Clustering 0.90                  0.12                       Random Forest  0.933759   
                                               0.15                       Random Forest  0.933759   
                                               0.20                       Random Forest  0.933759   
                         0.95                  0.07                       Random Forest  0.933759   
                                               0.10                       Random Forest  0.933759   
                                               0.15                       Random Forest  0.933759   
                         0.98                  0.03                       Random Forest  0.933759   
                                               0.05                       Random Forest  0.933759   
                                               0.10                       Random Forest  0.933759   
                         0.99                  0.02                       Random Forest  0.933759   
                                               0.04                       Random Forest  0.933759   
                                               0.08                       Random Forest  0.933759   

                                                                                         precision  \
experiment               correlation_threshold cluster_distance_threshold model                      
Correlation + Clustering 0.90                  0.12                       Random Forest        0.0   
                                               0.15                       Random Forest        0.0   
                                               0.20                       Random Forest        0.0   
                         0.95                  0.07                       Random Forest        0.0   
                                               0.10                       Random Forest        0.0   
                                               0.15                       Random Forest        0.0   
                         0.98                  0.03                       Random Forest        0.0   
                                               0.05                       Random Forest        0.0   
                                               0.10                       Random Forest        0.0   
                         0.99                  0.02                       Random Forest        0.0   
                     

### XGBoost

In [16]:
joint_tuning_grid = {
    0.90: [0.12, 0.15, 0.20],
    0.95: [0.07, 0.10, 0.15],
    0.98: [0.03, 0.05, 0.10],
    0.99: [0.02, 0.04, 0.08],
}

result_frames = []

for correlation_threshold, cluster_thresholds in joint_tuning_grid.items():

    cluster_methods = {
        str(cluster_threshold): cluster_threshold
        for cluster_threshold in cluster_thresholds
    }

    result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        models={"XGBoost": get_models()["XGBoost"]},
        experiment_name="Correlation + Clustering",
        methods=cluster_methods,
        parameter_name="cluster_distance_threshold",
        pipeline_kwargs={
            "imputer": "mean",
            "scaler": "standard",
            "missing_threshold": 0.6,
            "variance_threshold": 0.1,
            "correlation_threshold": correlation_threshold,
        },
    ).reset_index()

    result["correlation_threshold"] = correlation_threshold
    result["cluster_distance_threshold"] = (
        result["method"].astype(float)
    )

    result_frames.append(result)

xg_corr_clu_results = (
    pd.concat(result_frames, ignore_index=True)
    .drop(columns="method")
    .set_index(
        [
            "experiment",
            "correlation_threshold",
            "cluster_distance_threshold",
            "model",
        ]
    )
    .sort_index()
)

xg_corr_clu_results

threshold  \
experiment               correlation_threshold cluster_distance_threshold model                
Correlation + Clustering 0.90                  0.12                       XGBoost        0.5   
                                               0.15                       XGBoost        0.5   
                                               0.20                       XGBoost        0.5   
                         0.95                  0.07                       XGBoost        0.5   
                                               0.10                       XGBoost        0.5   
                                               0.15                       XGBoost        0.5   
                         0.98                  0.03                       XGBoost        0.5   
                                               0.05                       XGBoost        0.5   
                                               0.10                       XGBoost        0.5   
                         0.99                  0.02                       XGBoost        0.5   
                                               0.04                       XGBoost        0.5   
                                               0.08                       XGBoost        0.5   

                                                                                   accuracy  \
experiment               correlation_threshold cluster_distance_threshold model               
Correlation + Clustering 0.90                  0.12                       XGBoost  0.932163   
                                               0.15                       XGBoost  0.929769   
                                               0.20                       XGBoost  0.932961   
                         0.95                  0.07                       XGBoost  0.933759   
                                               0.10                       XGBoost  0.932961   
                                               0.15                       XGBoost  0.930567   
                         0.98                  0.03                       XGBoost  0.932163   
                                               0.05                       XGBoost  0.931365   
                                               0.10                       XGBoost  0.932163   
                         0.99                  0.02                       XGBoost  0.930567   
                                               0.04                       XGBoost  0.931365   
                                               0.08                       XGBoost  0.931365   

                                                                                   precision  \
experiment               correlation_threshold cluster_distance_threshold model                
Correlation + Clustering 0.90                  0.12                       XGBoost   0.375000   
                                               0.15                       XGBoost   0.142857   
                                               0.20                       XGBoost   0.333333   
                         0.95                  0.07                       XGBoost   0.500000   
                                               0.10                       XGBoost   0.333333   
                                               0.15                       XGBoost   0.000000   
                         0.98                  0.03                       XGBoost   0.250000   
                                               0.05                       XGBoost   0.200000   
                                               0.10                       XGBoost   0.333333   
                         0.99                  0.02                       XGBoost   0.000000   
                                               0.04                       XGBoost   0.000000   
                                               0.08                       XGBoost   0.200000   

                                                              

### Observations

- 상관계수와 군집화를 함께 적용하여 데이터의 중복성을 정제한 실험 결과를 관찰하였습니다.

- 공정 변수 간의 관계가 밀접한 것들을 정리하면서 분석에 사용된 원래 변수의 개수가 크게 줄어들었습니다. 이전 분산 제거 단계에서 370개 수준이던 원본 변수가 이번 단계를 거치며 약 160개에서 250개 사이로 알맞게 정제되었습니다.

- Logistic Regression의 경우, 상관계수 기준 0.90과 군집화 거리 기준 0.20 조건에서 원본 변수가 163.2개까지 줄어들었습니다. 이 조건에서 오검출은 66건, 미검출은 66건으로 서로 균형을 이루며 안정적인 예측 성능을 보여주었습니다.

- Random Forest의 경우, 데이터의 극심한 불균형으로 인해 기본 분류 기준인 0.5 하에서는 실제 불량을 전혀 잡아내지 못해 미검출이 83건으로 나타났습니다. 하지만 상관계수 기준 0.95와 군집화 거리 기준 0.15 조건에서 잠재적 분류 성능을 나타내는 ROC AUC 지표가 0.7132로 가장 높게 측정되었습니다.

- XGBoost 모델은 전반적으로 세 모델 중 가장 뛰어난 예측 잠재력을 보여주었습니다. 특히 상관계수 기준 0.95와 군집화 거리 기준 0.07 조건에서 전체 성능 지표인 ROC AUC가 0.7236으로 이번 실험 중 가장 높은 값을 기록했습니다. 이 조건에서 오검출은 단 2건에 불과했고 미검출은 81건을 나타냈습니다.

### Interpretation

- 서로 비슷한 정보를 담고 있는 공정 변수들을 하나로 묶고 중복된 변수를 제거한 결과, 분석에 사용되는 변수의 개수가 절반 가까이 줄어들었습니다. 그럼에도 성능이 떨어지지 않고 유지되거나 상승한 것은 불필요한 중복 노이즈가 성공적으로 제거되었음을 의미합니다.

- Logistic Regression에서 변수 개수가 가장 적게 남았을 때 오검출과 미검출이 동일하게 66건으로 잡힌 것은, 변수 간의 중복 정보를 정제하면서 모델의 불량 분류 판단이 한층 더 명확해졌기 때문으로 보입니다.

- Random Forest와 XGBoost 모델이 중복을 제거한 후에도 높은 수준의 ROC AUC 성능을 유지한 것은 핵심 불량 패턴을 학습할 수 있는 뼈대 데이터가 잘 마련되었음을 보여줍니다.

- 다만, 두 트리 모델 모두 0.5라는 기본 임계값 조건에서는 정상 제품으로만 예측하려는 편향된 모습을 보여 실제 불량을 놓치는 미검출 수가 여전히 많습니다. 이는 불량을 놓치는 품질 비용과 정상 제품을 불량으로 잘못 판정해 다시 검사하는 생산 비용 사이에서 균형을 맞추기 위한 추가 조율이 반드시 필요함을 시사합니다.

### Decision

- Logistic Regression의 최적 설정값은 상관계수 0.90, 군집화 거리 0.20으로 확정합니다. 원래 변수를 163.2개로 크게 줄이면서도 오검출과 미검출의 균형을 가장 잘 잡은 최적의 지점입니다.

- Random Forest의 최적 설정값은 상관계수 0.95, 군집화 거리 0.15로 결정합니다. 변수 개수를 177.2개 수준으로 다이어트하면서도 잠재적인 성능 지표인 ROC AUC를 0.7132까지 가장 우수하게 확보했기 때문입니다.

- XGBoost 모델의 최적 설정값은 상관계수 0.95, 군집화 거리 0.07로 결정합니다. 이번 실험 전체를 통틀어 가장 높은 예측 잠재력인 ROC AUC 0.7236을 확보하였고, 정상 제품을 오진해 재검사 비용을 쓰는 오검출 리스크를 단 2건으로 차단했기 때문입니다.

### Candidate

Logistic Regression : 0.90 - 0.20  
Random Forest : 0.95 - 0.15  
XGBoost : 0.95 - 0.07

## 6. Sampling

### Logistic Regression

In [ ]:
samplers = {
    "Baseline": None,
    "SMOTE": SMOTE(random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
}

lr_sampler_results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    models={"Logistic Regression": get_models()["Logistic Regression"]},
    experiment_name="sampler",
    methods=samplers,
    parameter_name="sampler",
    pipeline_kwargs={
        "imputer": "mean",
        "scaler": "standard",
        "missing_threshold": 0.2,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.90,
        "cluster_distance_threshold": 0.20
    },
)

lr_sampler_results

threshold  accuracy  \
experiment method          model                                      
sampler    Baseline        Logistic Regression        0.5  0.894653   
           SMOTE           Logistic Regression        0.5  0.839585   
           BorderlineSMOTE Logistic Regression        0.5  0.848364   
           ADASYN          Logistic Regression        0.5  0.837191   

                                                precision    recall        f1  \
experiment method          model                                                
sampler    Baseline        Logistic Regression   0.204819  0.204819  0.204819   
           SMOTE           Logistic Regression   0.148810  0.301205  0.199203   
           BorderlineSMOTE Logistic Regression   0.154839  0.289157  0.201681   
           ADASYN          Logistic Regression   0.142012  0.289157  0.190476   

                                                      f2  \
experiment method          model                           
sampler    Baseline        Logistic Regression  0.204819   
           SMOTE           Logistic Regression  0.250000   
           BorderlineSMOTE Logistic Regression  0.246407   
           ADASYN          Logistic Regression  0.239521   

                                                       confusion_matrix  \
experiment method          model                                          
sampler    Baseline        Logistic Regression   [[1104, 66], [66, 17]]   
           SMOTE           Logistic Regression  [[1027, 143], [58, 25]]   
           BorderlineSMOTE Logistic Regression  [[1039, 131], [59, 24]]   
           ADASYN          Logistic Regression  [[1025, 145], [59, 24]]   

                                                overkill_count  escape_count  \
experiment method          model                                               
sampler    Baseline        Logistic Regression              66            66   
           SMOTE           Logistic Regression             143            58   
           BorderlineSMOTE Logistic Regression             131            59   
           ADASYN          Logistic Regression             145            59   

                                                 roc_auc    pr_auc  \
experiment method          model                                     
sampler    Baseline        Logistic Regression  0.650922  0.129788   
           SMOTE           Logistic Regression  0.645093  0.128158   
           BorderlineSMOTE Logistic Regression  0.645855  0.125976   
           ADASYN          Logistic Regression  0.645948  0.128484   

                                                remaining_features  \
experiment method          model                                     
sampler    Baseline        Logistic Regression               534.8   
           SMOTE           Logistic Regression               534.8   
           BorderlineSMOTE Logistic Regression               534.8   
           ADASYN          Logistic Regression               534.8   

                                                remaining_features_std  \
experiment method          model                                         
sampler    Baseline        Logistic Regression               28.638436   
           SMOTE           Logistic Regression               28.638436   
           BorderlineSMOTE Logistic Regression               28.638436   
           ADASYN          Logistic Regression               28.638436   

                                                remaining_original_features  \
experiment method          model                                              
sampler    Baseline        Logistic Regression                        163.2   
           SMOTE           Logistic Regression                        163.2   
           BorderlineSMOTE Logistic Regression                        163.2   
           ADASYN          Logistic Regression                        163.2   

                                                remaining_original_features_std

### Random Forest

In [18]:
samplers = {
    "Baseline": None,
    "SMOTE": SMOTE(random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
}

rf_sampler_results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    models={"Random Forest": get_models()["Random Forest"]},
    experiment_name="sampler",
    methods=samplers,
    parameter_name="sampler",
    pipeline_kwargs={
        "imputer": "mean",
        "scaler": "standard",
        "missing_threshold": 0.6,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.95,
        "cluster_distance_threshold": 0.15
    },
)

rf_sampler_results

threshold  accuracy  precision  \
experiment method          model                                           
sampler    Baseline        Random Forest        0.5  0.933759        0.0   
           SMOTE           Random Forest        0.5  0.933759        0.5   
           BorderlineSMOTE Random Forest        0.5  0.930567        0.0   
           ADASYN          Random Forest        0.5  0.931365        0.0   

                                            recall        f1        f2  \
experiment method          model                                         
sampler    Baseline        Random Forest  0.000000  0.000000  0.000000   
           SMOTE           Random Forest  0.036145  0.067416  0.044379   
           BorderlineSMOTE Random Forest  0.000000  0.000000  0.000000   
           ADASYN          Random Forest  0.000000  0.000000  0.000000   

                                              confusion_matrix  \
experiment method          model                                 
sampler    Baseline        Random Forest  [[1170, 0], [83, 0]]   
           SMOTE           Random Forest  [[1167, 3], [80, 3]]   
           BorderlineSMOTE Random Forest  [[1166, 4], [83, 0]]   
           ADASYN          Random Forest  [[1167, 3], [83, 0]]   

                                          overkill_count  escape_count  \
experiment method          model                                         
sampler    Baseline        Random Forest               0            83   
           SMOTE           Random Forest               3            80   
           BorderlineSMOTE Random Forest               4            83   
           ADASYN          Random Forest               3            83   

                                           roc_auc    pr_auc  \
experiment method          model                               
sampler    Baseline        Random Forest  0.713207  0.153265   
           SMOTE           Random Forest  0.722475  0.179571   
           BorderlineSMOTE Random Forest  0.707291  0.151866   
           ADASYN          Random Forest  0.728632  0.183436   

                                          remaining_features  \
experiment method          model                               
sampler    Baseline        Random Forest               548.8   
           SMOTE           Random Forest               548.8   
           BorderlineSMOTE Random Forest               548.8   
           ADASYN          Random Forest               548.8   

                                          remaining_features_std  \
experiment method          model                                   
sampler    Baseline        Random Forest               29.362561   
           SMOTE           Random Forest               29.362561   
           BorderlineSMOTE Random Forest               29.362561   
           ADASYN          Random Forest               29.362561   

                                          remaining_original_features  \
experiment method          model                                        
sampler    Baseline        Random Forest                        177.2   
           SMOTE           Random Forest                        177.2   
           BorderlineSMOTE Random Forest                        177.2   
           ADASYN          Random Forest                        177.2   

                                          remaining_original_features_std  \
experiment method          model                                            
sampler    Baseline        Random Forest                         2.135416   
           SMOTE           Random Forest                         2.135416   
           BorderlineSMOTE Random Forest                         2.135416   
           ADASYN          Random Forest                         2.135416   

                                          added_indicator_features  \
experiment method          model                                     
sampler    Baseline        Random Forest                     371.6   
       

### XGBoost

In [21]:
samplers = {
    "Baseline": None,
    "SMOTE": SMOTE(random_state=42),
    "BorderlineSMOTE": BorderlineSMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
}

xg_sampler_results = run_experiment(
    X_train=X_train,
    y_train=y_train,
    models={"XGBoost": get_models()["XGBoost"]},
    experiment_name="sampler",
    methods=samplers,
    parameter_name="sampler",
    pipeline_kwargs={
        "imputer": "mean",
        "scaler": "standard",
        "missing_threshold": 0.6,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.95,
        "cluster_distance_threshold": 0.07
    },
)

xg_sampler_results

threshold  accuracy  precision    recall  \
experiment method          model                                               
sampler    Baseline        XGBoost        0.5  0.933759   0.500000  0.024096   
           SMOTE           XGBoost        0.5  0.924980   0.238095  0.060241   
           BorderlineSMOTE XGBoost        0.5  0.931365   0.440000  0.132530   
           ADASYN          XGBoost        0.5  0.929769   0.380952  0.096386   

                                          f1        f2  \
experiment method          model                         
sampler    Baseline        XGBoost  0.045977  0.029762   
           SMOTE           XGBoost  0.096154  0.070822   
           BorderlineSMOTE XGBoost  0.203704  0.154062   
           ADASYN          XGBoost  0.153846  0.113314   

                                          confusion_matrix  overkill_count  \
experiment method          model                                             
sampler    Baseline        XGBoost    [[1168, 2], [81, 2]]               2   
           SMOTE           XGBoost   [[1154, 16], [78, 5]]              16   
           BorderlineSMOTE XGBoost  [[1156, 14], [72, 11]]              14   
           ADASYN          XGBoost   [[1157, 13], [75, 8]]              13   

                                    escape_count   roc_auc    pr_auc  \
experiment method          model                                       
sampler    Baseline        XGBoost            81  0.723674  0.177500   
           SMOTE           XGBoost            78  0.707445  0.182909   
           BorderlineSMOTE XGBoost            72  0.707352  0.196880   
           ADASYN          XGBoost            75  0.700886  0.190624   

                                    remaining_features  \
experiment method          model                         
sampler    Baseline        XGBoost               569.0   
           SMOTE           XGBoost               569.0   
           BorderlineSMOTE XGBoost               569.0   
           ADASYN          XGBoost               569.0   

                                    remaining_features_std  \
experiment method          model                             
sampler    Baseline        XGBoost               29.873065   
           SMOTE           XGBoost               29.873065   
           BorderlineSMOTE XGBoost               29.873065   
           ADASYN          XGBoost               29.873065   

                                    remaining_original_features  \
experiment method          model                                  
sampler    Baseline        XGBoost                        197.4   
           SMOTE           XGBoost                        197.4   
           BorderlineSMOTE XGBoost                        197.4   
           ADASYN          XGBoost                        197.4   

                                    remaining_original_features_std  \
experiment method          model                                      
sampler    Baseline        XGBoost                         1.356466   
           SMOTE           XGBoost                         1.356466   
           BorderlineSMOTE XGBoost                         1.356466   
           ADASYN          XGBoost                         1.356466   

                                    added_indicator_features  \
experiment method          model                               
sampler    Baseline        XGBoost                     371.6   
           SMOTE           XGBoost                     371.6   
           BorderlineSMOTE XGBoost                     371.6   
           ADASYN          XGBoost                     371.6   

                                    added_indicator_features_std  
experiment method          model                                  
sampler    Baseline        XGBoost                     29.131426  
           SMOTE           XGBoost                     29.131426  
           BorderlineSMOTE XGBoost                     29.131426  
           ADASYN          XGBoo

### Observation

- 샘플러를 적용하여 불균형한 데이터를 보정한 실험 결과를 관찰하였습니다. 불량 데이터를 통계적으로 채워 넣는 작업이 진행되면서, 모델들의 실제 불량 검출력인 recall 지표에 유의미한 변화가 발생했습니다.

- Logistic Regression의 경우, smote를 적용했을 때 불량을 놓쳐 유출되는 미검출 수가 기존 66건에서 58건으로 줄어들며 가장 높은 검출 성능인 recall 0.3012를 기록했습니다. 다만 정상 다이를 불량으로 오진하는 오검출은 기존 66건에서 143건으로 대폭 증가했습니다.

- Random Forest은 여전히 기본 임계치 0.5 하에서 대부분의 불량을 잡아내지 못하는 한계를 보였습니다. 그래도 adasyn을 적용했을 때 잠재 성능 지표인 roc auc가 0.7286으로 이번 실험 중 Random Forest에서 가장 높게 측정되었습니다.

- xgboost 모델에서는 borderline smote를 적용했을 때 가장 고무적인 변화가 관찰되었습니다. 미검출 수가 기존 81건에서 72건으로 감소하여 recall 0.1325를 기록하였고, 오검출은 14건 수준으로 억제하며 전체적인 지표의 균형을 유지했습니다.

### Interpretation

- 불량률이 극히 낮은 반도체 공정 특성상, 인위적으로 불량 데이터를 합성해 늘려주면 모델이 불량의 특징을 더 적극적으로 학습하게 됩니다. 로지스틱 회귀에서 미검출이 감소한 대신 오검출이 급증한 것은, 데이터를 불리면서 판단 기준선이 정상 영역으로 넓게 확장되었기 때문으로 해석됩니다.

- Random Forest가 여전히 불량을 잡지 못하는 것은 가상으로 만든 데이터가 트리 분기점을 확실하게 자극하지 못했기 때문으로 보입니다. 하지만 adasyn 조건에서 roc auc가 상승한 것은 데이터 분포의 밀도를 세밀하게 조정하여 분류 경계를 그리는 잠재력을 다소 향상시켰음을 뜻합니다.

- xgboost 모델에서 borderline smote가 가장 효과적이었던 이유는, 정상 제품과 불량 제품의 모호한 경계선에 있는 데이터를 중점적으로 학습시켰기 때문으로 분석됩니다. 공정상 미세한 변동으로 판정이 흐릿해지는 다이들을 더 정밀하게 판단할 수 있도록 뼈대를 잡아준 결과입니다.

### Decision

- Logistic Regression의 최적 설정값은 추가 샘플링을 적용하지 않은 베이스라인(Baseline)으로 확정합니다. SMOTE 적용 시 실제 불량 검출은 단 8건 늘어나는 데 그친 반면 오검출이 77건이나 폭증하여, 불량 1건을 더 잡기 위해 정상 다이 9.6건을 희생해야 하는 심각한 생산성 저하를 확인했기 때문입니다.

- Random Forest의 최적 설정값은 adasyn으로 선정합니다. 오검출을 최소화하면서도 모델의 전반적인 잠재 예측력을 뜻하는 roc auc를 0.7286까지 가장 효과적으로 끌어올렸기 때문입니다.

- XGBoost 모델의 최적 설정값은 borderline smote로 선정합니다. 무작위로 데이터를 채워 넣는 일반 smote에 비해 오검출을 단 14건으로 효과적으로 통제하면서 실제 불량 검출률을 높이는 최적의 균형을 확보했습니다.

### Candidate

Logistic Regression : Baseline  
Random Forest : ADASYN  
XGBoost : BorderlineSMOTE

## 7. Hyperparameter Tuning & Test

### Evaluation Metrics: PR-AUC

In [105]:
TUNING_GRIDS = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1.0],
        "model__penalty": ["l2"]
    },
    "Random Forest": {
        "model__max_depth": [4, 6, 8],
        "model__min_samples_leaf": [8, 16],
        "model__max_features": ["sqrt"]
    },
    "XGBoost": {
        "model__max_depth": [3, 4, 5],
        "model__min_child_weight": [5, 10],
        "model__reg_lambda": [1.0, 5.0, 10.0]
    }
}

configs = {
    "Logistic Regression": {
        "missing_threshold": 0.20,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.90,
        "cluster_distance_threshold": 0.20,
        "sampler": None,
    },
    "Random Forest": {
        "missing_threshold": 0.60,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.95,
        "cluster_distance_threshold": 0.15,
        "sampler": ADASYN(random_state=42),
    },
    "XGBoost": {
        "missing_threshold": 0.60,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.95,
        "cluster_distance_threshold": 0.07,
        "sampler": BorderlineSMOTE(random_state=42),
    }
}

models = get_models()

tuned_pipelines = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model_info in models.items():
    if model_name not in configs or model_name not in TUNING_GRIDS:
        continue
        
    cfg = configs[model_name]
    param_grid = TUNING_GRIDS[model_name]
    
    base_pipeline = build_pipeline(
        model=model_info,
        imputer="mean",
        scaler="standard",
        missing_threshold=cfg["missing_threshold"],
        variance_threshold=cfg["variance_threshold"],
        correlation_threshold=cfg["correlation_threshold"],
        cluster_distance_threshold=cfg["cluster_distance_threshold"],
        sampler=cfg["sampler"]
    )
    
    print(f"=== {model_name} 튜닝 시작 (평가지표: PR-AUC) ===")

    grid_search = GridSearchCV(
        estimator=base_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="average_precision", 
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"최적 파라미터: {grid_search.best_params_}")
    print(f"최고 PR-AUC Score: {grid_search.best_score_:.4f}\n")
    
    tuned_pipelines[model_name] = grid_search.best_estimator_

=== Logistic Regression 튜닝 시작 (평가지표: PR-AUC) ===
Fitting 5 folds for each of 3 candidates, totalling 15 fits
최적 파라미터: {'model__C': 0.1, 'model__penalty': 'l2'}
최고 PR-AUC Score: 0.1731

=== Random Forest 튜닝 시작 (평가지표: PR-AUC) ===
Fitting 5 folds for each of 6 candidates, totalling 30 fits
최적 파라미터: {'model__max_depth': 6, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 8}
최고 PR-AUC Score: 0.1991

=== XGBoost 튜닝 시작 (평가지표: PR-AUC) ===
Fitting 5 folds for each of 18 candidates, totalling 90 fits
최적 파라미터: {'model__max_depth': 4, 'model__min_child_weight': 5, 'model__reg_lambda': 1.0}
최고 PR-AUC Score: 0.2458



### Logistic Regression

In [ ]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
lr_hy_pipeline = tuned_pipelines["Logistic Regression"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
train_proba = lr_hy_pipeline.predict_proba(X_train)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
lr_hy_train_metrics = calculate_metrics(
    y_true=y_train,
    y_proba=train_proba,
)

pd.DataFrame([lr_hy_train_metrics])

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,0.955307,0.909091,0.361446,0.517241,0.410959,"[[1167, 3], [53, 30]]",3,53,0.956637,0.717241


### Random Forest

In [107]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
rf_hy_pipeline = tuned_pipelines["Random Forest"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
train_proba = rf_hy_pipeline.predict_proba(X_train)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
rf_hy_train_metrics = calculate_metrics(
    y_true=y_train,
    y_proba=train_proba,
)

pd.DataFrame([rf_hy_train_metrics])

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,0.94174,0.565789,0.518072,0.540881,0.526961,"[[1137, 33], [40, 43]]",33,40,0.961693,0.676288


### XGBoost

In [108]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
xgb_hy_pipeline = tuned_pipelines["XGBoost"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
train_proba = xgb_hy_pipeline.predict_proba(X_train)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
xg_hy_train_metrics = calculate_metrics(
    y_true=y_train,
    y_proba=train_proba,
)

pd.DataFrame([xg_hy_train_metrics])

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,1.0,1.0,1.0,1.0,1.0,"[[1170, 0], [0, 83]]",0,0,1.0,1.0


### Observations

- Logistic Regression은 정확도 0.955307, ROC-AUC 0.956637, PR-AUC 0.717241을 기록했습니다. 오검출(Overkill)은 3건으로 매우 낮았으나, 불량이 정상으로 분류되어 통과되는 미검출(Escape)은 53건이 발생해 재현율(Recall)이 0.361446에 그쳤습니다.

- Random Forest는 정확도 0.94174, ROC-AUC 0.961693, PR-AUC 0.676288을 보였습니다. 오검출은 33건이었으며, 미검출은 40건으로 재현율은 0.518072를 기록했습니다.

- XGBoost는 정확도, 정밀도, 재현율, F1, F2 지표와 ROC-AUC, PR-AUC까지 모든 평가 지표에서 1.0을 달성했습니다. 오검출과 미검출 모두 0건으로 학습 데이터를 완벽하게 예측하는 양상을 보였습니다.

### Interpretation

- XGBoost가 학습 데이터에서 모든 지표가 1.0으로 수렴한 것은 전형적인 Overfitting 상태입니다. 이는 모델이 불량 데이터의 본질적이고 일반적인 특성을 학습했다기보다는 학습 데이터 내의 미세한 노이즈와 아웃라이어 분포까지 기계적으로 암기했음을 의미합니다.

- Logistic Regression과 Random Forest 역시 XGBoost에 비해 정도가 약하긴 하지만 샘플링 비교 결과와 비교해 보았을 때 모델 성능에 비약적인 향상이 있었던 것으로 보아 과적합이 발생하고 있다고 판단했습니다.

- 학습 데이터에 완벽히 피팅된 모델은 공정 변동 속에서 강건성을 잃고, 후속 양산 데이터가 투입되었을 때 대규모 오검출이나 미검출 품질 사고로 이어질 가능성이 매우 큽니다.

### Decision

- 학습 단계에서 극단적인 Overfitting이 관찰된 XGBoost 모델을 디버깅하기 위해, 트리의 최대 깊이(max_depth)를 한층 제한하여 모델의 복잡도를 낮추고, L2 정규화 규제(reg_lambda)를 대폭 강화하는 방향으로 하이퍼파라미터 그리드를 재설정하여 재학습을 집행하겠습니다.

- 내부적인 일반화 성능 하락이 확인된 Random Forest 모델의 하이퍼파라미터 그리드도 함께 디버깅하겠습니다. Random Forest는 트리의 최대 깊이를 조절하고 리프 노드의 최소 샘플 수(min_samples_leaf) 기준을 높여 소수 불량 노이즈에 대한 과도한 분할을 억제하겠습니다.

## 8. Re-tuning Hyperparameters

In [116]:
TUNING_GRIDS = {
    "Logistic Regression": {
        "model__C": [0.001, 0.01, 0.1],
        "model__penalty": ["elasticnet"],
        "model__solver": ["saga"],
        "model__l1_ratio": [0.1, 0.5]
    },
    "Random Forest": {
        "model__max_depth": [3, 4, 5],
        "model__min_samples_leaf": [16, 32],
        "model__min_samples_split": [10, 20],
        "model__max_features": ["sqrt"],
        "model__class_weight": ["balanced"]
    },
    "XGBoost": {
        "model__max_depth": [2, 3, 4],
        "model__min_child_weight": [10, 20],
        "model__reg_lambda": [5.0, 10.0, 20.0],
        "model__reg_alpha": [0, 0.1, 1.0],
        "model__subsample": [0.6, 0.8],
        "model__colsample_bytree": [0.6, 0.8],
        "model__learning_rate": [0.01, 0.05, 0.1]
    }
}

configs = {
    "Logistic Regression": {
        "missing_threshold": 0.20,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.90,
        "cluster_distance_threshold": 0.20,
        "sampler": None,
    },
    "Random Forest": {
        "missing_threshold": 0.60,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.95,
        "cluster_distance_threshold": 0.15,
        "sampler": ADASYN(random_state=42),
    },
    "XGBoost": {
        "missing_threshold": 0.60,
        "variance_threshold": 0.1,
        "correlation_threshold": 0.95,
        "cluster_distance_threshold": 0.07,
        "sampler": BorderlineSMOTE(random_state=42),
    }
}

models = get_models()

tuned_pipelines = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model_info in models.items():
    if model_name not in configs or model_name not in TUNING_GRIDS:
        continue
        
    cfg = configs[model_name]
    param_grid = TUNING_GRIDS[model_name]
    
    base_pipeline = build_pipeline(
        model=model_info,
        imputer="mean",
        scaler="standard",
        missing_threshold=cfg["missing_threshold"],
        variance_threshold=cfg["variance_threshold"],
        correlation_threshold=cfg["correlation_threshold"],
        cluster_distance_threshold=cfg["cluster_distance_threshold"],
        sampler=cfg["sampler"]
    )
    
    print(f"=== {model_name} 튜닝 시작 (평가지표: PR-AUC) ===")

    grid_search = GridSearchCV(
        estimator=base_pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="average_precision", 
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"최적 파라미터: {grid_search.best_params_}")
    print(f"최고 PR-AUC Score: {grid_search.best_score_:.4f}\n")
    
    tuned_pipelines[model_name] = grid_search.best_estimator_

=== Logistic Regression 튜닝 시작 (평가지표: PR-AUC) ===
Fitting 5 folds for each of 6 candidates, totalling 30 fits
최적 파라미터: {'model__C': 0.1, 'model__l1_ratio': 0.5, 'model__penalty': 'elasticnet', 'model__solver': 'saga'}
최고 PR-AUC Score: 0.1744

=== Random Forest 튜닝 시작 (평가지표: PR-AUC) ===
Fitting 5 folds for each of 12 candidates, totalling 60 fits
최적 파라미터: {'model__class_weight': 'balanced', 'model__max_depth': 4, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 32, 'model__min_samples_split': 10}
최고 PR-AUC Score: 0.1891

=== XGBoost 튜닝 시작 (평가지표: PR-AUC) ===
Fitting 5 folds for each of 648 candidates, totalling 3240 fits
최적 파라미터: {'model__colsample_bytree': 0.6, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__min_child_weight': 10, 'model__reg_alpha': 0, 'model__reg_lambda': 5.0, 'model__subsample': 0.8}
최고 PR-AUC Score: 0.2509



### Logistic Regression

In [117]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
lr_hy_pipeline = tuned_pipelines["Logistic Regression"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
train_proba = lr_hy_pipeline.predict_proba(X_train)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
lr_hy_train_metrics = calculate_metrics(
    y_true=y_train,
    y_proba=train_proba,
)

pd.DataFrame([lr_hy_train_metrics])

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,0.940942,1.0,0.108434,0.195652,0.131965,"[[1170, 0], [74, 9]]",0,74,0.913943,0.572377


### Random Forest

In [118]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
rf_hy_pipeline = tuned_pipelines["Random Forest"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
train_proba = rf_hy_pipeline.predict_proba(X_train)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
rf_hy_train_metrics = calculate_metrics(
    y_true=y_train,
    y_proba=train_proba,
)

pd.DataFrame([rf_hy_train_metrics])

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,0.892259,0.283333,0.409639,0.334975,0.376106,"[[1084, 86], [49, 34]]",86,49,0.860931,0.407055


### XGBoost

In [119]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
xgb_hy_pipeline = tuned_pipelines["XGBoost"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
train_proba = xgb_hy_pipeline.predict_proba(X_train)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
xg_hy_train_metrics = calculate_metrics(
    y_true=y_train,
    y_proba=train_proba,
)

pd.DataFrame([xg_hy_train_metrics])

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,0.978452,0.982759,0.686747,0.808511,0.730769,"[[1169, 1], [26, 57]]",1,26,0.992091,0.929963


### Observations

- 교차 검증(Cross-Validation) 과정에서의 최고 PR-AUC 평가지표는 XGBoost가 0.2509로 가장 우수하였으며, Random Forest가 0.1891, Logistic Regression이 0.1744 순으로 나타났습니다.

- 튜닝이 완료된 Logistic Regression의 훈련 데이터 성능은 정확도 0.940942, ROC-AUC 0.913943, PR-AUC 0.572377을 기록했습니다. 오검출은 0건으로 통제되었으나, 미검출은 74건이 발생하여 재현율이 0.108434 수준에 머물렀습니다.

- Random Forest의 훈련 데이터 성능은 정확도 0.892259, ROC-AUC 0.860931, PR-AUC 0.407055를 기록했습니다. 오검출은 86건, 미검출은 49건이었으며 재현율은 0.409639였습니다.

- XGBoost의 훈련 데이터 성능은 정확도 0.978452, ROC-AUC 0.992091, PR-AUC 0.929963을 기록했습니다. 오검출은 1건으로 최소화되었고, 미검출은 26건으로 감소하여 재현율 0.686747을 확보했습니다.

### Interpretation

- 이전 실험에서 관찰되었던 XGBoost의 극단적인 과적합(훈련 데이터 지표 모두 1.0) 상태가 하이퍼파라미터 제어를 통해 일부분 해소되었다고 보입니다. 최대 깊이를 3으로 제한하고, L2 정규화 규제인 reg_lambda를 5.0으로 적용하는 등 모델의 표현 용량을 억제한 조치가 유효했습니다. 훈련 데이터 기준 PR-AUC가 0.929963으로 하향 조정된 것은 모델이 학습용 데이터의 개별 노이즈를 맹목적으로 암기하는 것을 멈추고 일반적인 결정 경계를 그리기 시작했다는 긍정적인 신호입니다.

- 그럼에도 불구하고 모든 모델에서 훈련 데이터 성능과 교차 검증 점수(XGBoost 기준 Train PR-AUC 0.929963 대 CV PR-AUC 0.2509) 사이에 여전히 격차가 존재합니다. 이는 ADASYN을 통해 학습 데이터 내에 인위적으로 생성된 불량 샘플의 밀도와 실제 교차 검증 폴드 내부의 불균형한 데이터 분포 간에 근본적인 불일치가 존재하기 때문입니다. 즉, 모델이 합성 데이터에 반응하는 수준 대비 일반화 성능은 여전히 일정 부분 제약을 받고 있음을 뜻합니다.

- Random Forest는 오검출이 86건까지 치솟아 멀쩡한 칩을 폐기함으로써 발생하는 수율 손실 비용이 너무 큽니다. 따라서 0.5라는 고정된 임계값 대신, 합격과 불합격을 판정하는 기준 마진(임계값)을 유연하게 조율하는 가드밴드 설계가 필수적입니다.

### Decision

- 하이퍼파라미터 규제를 통해 모든 모델의 뼈대(아키텍처)를 확정하였으므로, 이 모델들을 기반으로 본격적인 분류 임계값 시뮬레이션을 수행하겠습니다. 데이터 누설이 없는 상태에서 모델의 실제 예측 확률 밀도를 확보하기 위해, 교차 검증 과정에서 도출된 아웃 오브 폴드(OOF) 예측 확률을 수집하겠습니다.

- 수집된 OOF 예측 확률 분포 위에서 분류 임계값을 0.05부터 0.90까지 세밀하게 변경해 가며 오검출과 미검출의 추이를 정량적으로 분석하겠습니다. 이를 통해 미검출 품질 사고를 완벽히 통제하는 동시에 오검출로 인한 수율 손실을 최소화하는 최적의 가드밴드 임계값을 각 모델별로 확정하겠습니다.

- 학습 및 검증 과정에 전혀 관여하지 않고 격리해 두었던 최종 테스트 데이터셋을 최종 단계에서 단 한 번 대입하겠습니다. 미리 설계한 모델과 최적 임계값을 최종 테스트 데이터에 Blind 조건으로 적용하여, 실제 양산 라인에 배포했을 때 확보할 수 있는 객관적인 종합 수율 및 품질 신뢰성을 최종 검증하겠습니다.

## 9. Classification Threshold

In [121]:
threshold_experiment_df = run_cv_threshold_experiment(
    tuned_pipelines=tuned_pipelines,
    X_train=X_train,
    y_train=y_train,
    cv=cv
)

threshold_experiment_df

,model,threshold,accuracy,precision,recall,f1,f2,overkill_count,escape_count,roc_auc,pr_auc
0,Logistic Regression,0.05,0.615323,0.114120,0.710843,0.196667,0.347468,458,24,0.709608,0.144543
1,Logistic Regression,0.10,0.810854,0.153153,0.409639,0.222951,0.306859,188,49,0.709608,0.144543
2,Logistic Regression,0.15,0.870710,0.184000,0.277108,0.221154,0.251641,102,60,0.709608,0.144543
3,Logistic Regression,0.20,0.897047,0.212500,0.204819,0.208589,0.206311,63,66,0.709608,0.144543
4,Logistic Regression,0.25,0.907422,0.200000,0.132530,0.159420,0.142119,44,72,0.709608,0.144543
5,Logistic Regression,0.30,0.915403,0.171429,0.072289,0.101695,0.081744,29,77,0.709608,0.144543
6,Logistic Regression,0.35,0.918595,0.193548,0.072289,0.105263,0.082645,25,77,0.709608,0.144543
7,Logistic Regression,0.40,0.923384,0.217391,0.060241,0.094340,0.070423,18,78,0.709608,0.144543
8,Logistic Regression,0.45,0.923384,0.157895,0.036145,0.058824,0.042735,16,80,0.709608,0.144543
9,Logistic Regression,0.50,0.924980,0.076923,0.012048,0.020833,0.014493,12,82,0.709608,0.144543


### Observations

- 임계값을 낮출수록 세 모델 모두 재현율이 증가하고 오검출이 크게 증가하는 전형적인 Trade-off가 나타났습니다. 반대로 임계값을 높이면 정확도는 상승하지만 미검출이 증가했습니다.

- Logistic Regression은 임계값 0.20에서 F1 0.208589, F2 0.206311로 비교적 균형적인 성능을 보였으며, 임계값 0.05에서는 재현율 0.710843까지 확보할 수 있었습니다.

- Random Forest는 임계값 0.05~0.20에서 재현율이 0.96~1.00으로 매우 높았지만, 오검출이 최대 1,170건까지 발생하여 실제 불량 판별 모델로 사용하기에는 과도한 수준이었습니다.

- XGBoost는 임계값 0.35~0.50 구간에서 F1 약 0.28~0.29, F2 약 0.29~0.34로 가장 안정적인 균형을 보였습니다. 특히 임계값 0.35에서 F1 0.287425, F2 0.288462를 기록했습니다.

### Interpretation

- Random Forest는 ROC-AUC가 0.673으로 상대적으로 낮고, 임계값에 따른 오검출 부담도 커 현재 조건에서는 우선순위가 낮습니다.

- Logistic Regression은 ROC-AUC 0.710, PR-AUC 0.145로 기본적인 불량 식별 능력은 있으나, 높은 재현율을 확보하려면 오검출이 급격히 증가합니다.

- XGBoost는 ROC-AUC 0.708, PR-AUC 0.217로 세 모델 중 가장 높은 PR-AUC를 기록하여 희소한 불량 클래스의 식별 능력에서 가장 유리한 모습을 보였습니다.

### Decision

- Logistic Regression: threshold = 0.20
    - F1 = 0.2086, F2 = 0.2063으로 0.20 부근에서 균형이 가장 양호했습니다.
    - Recall 0.2048을 확보하면서 Overkill 63건, Escape 66건으로 두 오류 유형의 균형도 비교적 좋았습니다.
    - 이후 threshold를 높이면 Accuracy는 증가하지만 Recall이 빠르게 감소하므로 0.20을 최종 임계값으로 선정했습니다.

- Random Forest: threshold = 0.50
    - F1 = 0.2234로 가장 높았으며, Recall 0.2530과 Precision 0.2000을 확보했습니다.
    - threshold 0.55 이상에서는 Recall이 급격히 감소하는 반면, 0.45 이하에서는 Overkill이 크게 증가했습니다.
    - 따라서 불량 검출과 오검출 사이의 균형이 가장 적절한 0.50을 선정했습니다.

- XGBoost: threshold = 0.30
    - Fprecision 0.2321, recall 0.3133, F1-score 0.2667, F2-score 0.2928을 기록하여 불량 검출과 오검출 사이의 균형이 가장 적절하다고 판단했습니다.
    - 특히 불량 미검출을 줄이는 목적을 고려할 때 0.40~0.50보다 높은 recall을 확보하면서도 지나치게 낮은 threshold에서 발생하는 오검출 증가를 어느 정도 억제할 수 있어 최종 threshold로 선정합니다.

## 10. Final Model Evaluation & Generalization Performance

### Logistic Regression

In [155]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
lr_hy_pipeline = tuned_pipelines["Logistic Regression"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
test_proba = lr_hy_pipeline.predict_proba(X_test)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
lr_hy_test_metrics = calculate_metrics(
    y_true=y_test,
    y_proba=test_proba,
    threshold=0.2,
)

lr_hy_test_result = pd.DataFrame([lr_hy_test_metrics])
lr_hy_test_result

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.2,0.894904,0.125,0.095238,0.108108,0.1,"[[279, 14], [19, 2]]",14,19,0.650252,0.139048


### Random Forest

In [156]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
rf_hy_pipeline = tuned_pipelines["Random Forest"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
test_proba = rf_hy_pipeline.predict_proba(X_test)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
rf_hy_test_metrics = calculate_metrics(
    y_true=y_test,
    y_proba=test_proba,
    threshold=0.5,
)

rf_hy_test_result = pd.DataFrame([rf_hy_test_metrics])
rf_hy_test_result

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,0.901274,0.291667,0.333333,0.311111,0.324074,"[[276, 17], [14, 7]]",17,14,0.786446,0.207579


### XGBoost

In [157]:
# 1. 튜닝과 전처리가 완료된 최적의 최종 XGBoost 파이프라인 가져오기
xgb_hy_pipeline = tuned_pipelines["XGBoost"]

# 2. 완전히 격리해 두었던 '진짜 실전 데이터' X_test로 확률 예측 수행
test_proba = xgb_hy_pipeline.predict_proba(X_test)[:, 1]

# 3. 우리가 결정한 최종 임계값 0.15와 calculate_metrics를 사용하여 일괄 평가
xg_hy_test_metrics = calculate_metrics(
    y_true=y_test,
    y_proba=test_proba,
    threshold=0.3,
)

xg_hy_test_result = pd.DataFrame([xg_hy_test_metrics])
xg_hy_test_result

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.3,0.904459,0.263158,0.238095,0.25,0.242718,"[[279, 14], [16, 5]]",14,16,0.72745,0.177114


### Observations

- Logistic Regression은 Test 데이터에서 정확도 0.894904, precision 0.125, recall 0.095238, F1 0.108108, ROC-AUC 0.650252, PR-AUC 0.139048을 기록했습니다. 오검출은 14건, 미검출은 19건으로 나타났습니다.

- Random Forest는 정확도 0.901274, precision 0.291667, recall 0.333333, F1 0.311111, ROC-AUC 0.786446, PR-AUC 0.207579를 기록했습니다. 오검출 17건, 미검출 14건으로 세 모델 중 가장 높은 불량 탐지 성능을 보였습니다.

- XGBoost는 정확도 0.904459, precision 0.263158, recall 0.238095, F1 0.250000, ROC-AUC 0.727450, PR-AUC 0.177114를 기록했습니다. 오검출 14건, 미검출 16건으로 나타났습니다.

- 세 모델 모두 Accuracy는 약 0.90 수준이지만, 불량 클래스의 비율이 낮기 때문에 Accuracy만으로 모델 성능을 판단하기에는 한계가 있습니다. 불균형 데이터에서는 precision-recall 계열 지표를 함께 확인하는 것이 중요합니다.

### Interpretation

- Random Forest가 최종 Test 데이터에서 가장 우수한 모델로 나타났습니다. 특히 ROC-AUC 0.786446, PR-AUC 0.207579, F1 0.311111로 세 모델 중 가장 높은 수준을 보였습니다.

- Random Forest는 21개의 실제 불량 중 7개를 탐지하여 recall 33.33%를 확보했으며, Logistic Regression의 2개, XGBoost의 5개보다 많았습니다.

- Logistic Regression은 threshold를 0.20으로 낮췄음에도 실제 불량 탐지가 2건에 그쳐 미검출 19건이 발생했습니다. 따라서 최종 불량 탐지 모델로서는 상대적으로 부족한 결과입니다.

- XGBoost는 Accuracy가 0.904459로 가장 높았지만, 불량 탐지 관점에서는 Random Forest보다 recall과 PR-AUC가 낮았습니다. 따라서 높은 Accuracy만으로 XGBoost를 최종 모델로 선택하는 것은 적절하지 않습니다.

- 특히 Random Forest의 Test PR-AUC가 0.207579로 가장 높았다는 점은 희소한 불량 클래스를 대상으로 한 실제 탐지 성능에서 의미 있는 결과입니다. PR-AUC/AP는 다양한 threshold에서 precision-recall 관계를 종합하는 지표이므로 불균형 문제에서 유용하게 볼 수 있습니다.

### Decision

- 최종 Test 평가 결과를 기준으로 Random Forest를 최종 불량 예측 모델로 선정합니다.

- 최종 Classification Threshold는 기존 실험에서 선정한 Logistic Regression = 0.20, Random Forest = 0.50, XGBoost = 0.30을 그대로 적용했습니다.

- 세 모델 중 Random Forest가 Recall 0.333333, F1 0.311111, ROC-AUC 0.786446, PR-AUC 0.207579로 가장 균형적인 불량 탐지 성능을 보였으므로 최종 모델로 결정합니다.

- 다만 실제 불량 21건 중 14건을 놓치고 있어 실무적인 불량 검출 시스템으로 사용하기에는 추가적인 threshold 조정이나 비용 민감 학습 등의 개선이 필요합니다. Threshold는 단순히 0.5를 사용하는 것이 아니라 실제 업무에서 false positive와 false negative의 비용을 고려하여 결정하는 것이 적절합니다.

In [158]:
final_result = pd.concat([lr_hy_test_result, rf_hy_test_result, xg_hy_test_result])
final_result.index = ["Logistic Regression", "Random Forest", "XGBoost"]
final_result

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
Logistic Regression,0.2,0.894904,0.125000,0.095238,0.108108,0.100000,"[[279, 14], [19, 2]]",14,19,0.650252,0.139048
Random Forest,0.5,0.901274,0.291667,0.333333,0.311111,0.324074,"[[276, 17], [14, 7]]",17,14,0.786446,0.207579
XGBoost,0.3,0.904459,0.263158,0.238095,0.250000,0.242718,"[[279, 14], [16, 5]]",14,16,0.727450,0.177114


In [159]:
rf_hy_test_result

,threshold,accuracy,precision,recall,f1,f2,confusion_matrix,overkill_count,escape_count,roc_auc,pr_auc
0,0.5,0.901274,0.291667,0.333333,0.311111,0.324074,"[[276, 17], [14, 7]]",17,14,0.786446,0.207579
